# Berlin-wide building classification (Path B, all 12 Bezirke)

Runs the same ALKIS-labeled Zhou et al. reproduction as `berlin_mitte.ipynb`'s Path B
(`clf2`), but city-wide instead of one borough at a time -- via `src/prepare_data.py`.
`berlin_mitte.ipynb` is left untouched as the single-borough baseline/reference.

OSM data comes from the local `data/raw/berlin-260805.osm.pbf` extract (via `pyrosm`)
instead of live per-borough Overpass calls; ALKIS ground truth comes from one paginated
whole-Berlin WFS call instead of a bbox-scoped one. Every buffer/spatial-join feature is
computed once on city-wide GeoDataFrames -- Bezirk boundaries only come in afterward as a
bookkeeping column, never as a filter on the feature computation (looping the pipeline
independently per Bezirk would undercount road/parking proximity at every internal border).

**Heads up before running:** this parses a 94MB PBF covering ~500k+ buildings and makes a
few live network calls (Bezirk boundaries + whole-Berlin ALKIS WFS, paginated). Expect this
to take a while the first time -- `build_feature_table` checkpoints each expensive step to
`data/raw/*.parquet` and the final table to `data/processed/berlin_buildings_features.parquet`,
so re-running this notebook after the first successful pass should be fast (skips straight to
loading the checkpoints).

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path("..").resolve()))

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

from src.prepare_data import build_feature_table, feature_columns

In [2]:
buildings = build_feature_table(
    pbf_path="../data/raw/berlin-260805.osm.pbf",
    cache_dir="../data/raw",
    out_path="../data/processed/berlin_buildings_features.parquet",
)
print(buildings.shape)
buildings["bezirk"].value_counts(dropna=False)

(537141, 99)


bezirk
Pankow                        66835
Steglitz-Zehlendorf           58105
Reinickendorf                 57122
Treptow-Köpenick              55748
Spandau                       45899
Tempelhof-Schöneberg          45482
Marzahn-Hellersdorf           44435
Neukölln                      40603
NaN                           33783
Lichtenberg                   26273
Charlottenburg-Wilmersdorf    25338
Mitte                         24888
Friedrichshain-Kreuzberg      12630
Name: count, dtype: int64

In [3]:
buildings["alkis_label"].value_counts(dropna=False)

alkis_label
residential        266121
NaN                205973
non_residential     65047
Name: count, dtype: int64

In [4]:
feature_cols = feature_columns(buildings)
print(len(feature_cols), "features")

labeled = buildings.dropna(subset=["alkis_label"])
X = labeled[feature_cols]
y = labeled["alkis_label"]

print(X.shape, y.value_counts())

86 features


(331168, 86) alkis_label
residential        266121
non_residential     65047
Name: count, dtype: int64


In [5]:
# Same hyperparameters as clf2 in berlin_mitte.ipynb -- verified against the paper's own
# repo (FFX_County_Extraction.ipynb) -- so this run is directly comparable to the existing
# Mitte-only baseline numbers in docs/phase4-findings.md (86.8%/87.4% accuracy).
# NOTE: the Path C detour in berlin_mitte.ipynb found these under-tuned for a larger feature
# space at Mitte's ~5k-row scale (min_impurity_decrease=0.001, min_samples_split=80 scored
# higher there) -- worth re-sweeping at this scale too as a follow-up, not done here to keep
# this first Berlin-wide number directly comparable to the documented baseline.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

clf2_berlin = DecisionTreeClassifier(
    class_weight={"non_residential": 2.0, "residential": 1.0},
    min_samples_split=20,
    min_impurity_decrease=0.0001,
    random_state=42,
)
clf2_berlin.fit(X_train, y_train)
y_pred = clf2_berlin.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.9189087175770753


                 precision    recall  f1-score   support

non_residential       0.79      0.80      0.79     13009
    residential       0.95      0.95      0.95     53225

       accuracy                           0.92     66234
      macro avg       0.87      0.87      0.87     66234
   weighted avg       0.92      0.92      0.92     66234



[[10397  2612]
 [ 2759 50466]]


In [6]:
# Per-bezirk breakdown -- cheap now that 'bezirk' is a column. Worth checking whether the
# accuracy gap vs. the paper's 96.7-98% concentrates in dense inner boroughs (Mitte,
# Friedrichshain-Kreuzberg) vs. more suburban outer ones (Spandau, Reinickendorf), which
# would be a quantitative confirmation of the mixed-use urban-form explanation
# docs/phase4-findings.md already gives for the gap.
test_bezirk = labeled.loc[X_test.index, "bezirk"]
results = pd.DataFrame({"bezirk": test_bezirk.values, "y_true": y_test.values, "y_pred": y_pred})

per_bezirk = results.groupby("bezirk").apply(
    lambda g: pd.Series({
        "n": len(g),
        "accuracy": accuracy_score(g["y_true"], g["y_pred"]),
        "non_residential_f1": f1_score(g["y_true"], g["y_pred"], pos_label="non_residential", zero_division=0),
    }),
    include_groups=False,
).sort_values("accuracy")
per_bezirk

,n,accuracy,non_residential_f1
bezirk,,,
Marzahn-Hellersdorf,6555.0,0.867887,0.682318
Spandau,6191.0,0.880472,0.688290
Friedrichshain-Kreuzberg,2061.0,0.902475,0.822281
Treptow-Köpenick,8165.0,0.902511,0.783224
Mitte,3338.0,0.909826,0.869978
Pankow,7140.0,0.910924,0.789960
Charlottenburg-Wilmersdorf,3303.0,0.927642,0.838840
Reinickendorf,7076.0,0.937535,0.813502
Lichtenberg,3106.0,0.942048,0.898075


In [7]:
pd.Series(clf2_berlin.feature_importances_, index=feature_cols).sort_values(ascending=False).head(15)

landuse_residential      0.421795
area_m2                  0.272418
has_name                 0.042423
building_commercial      0.029215
building_school          0.023140
landuse_allotments       0.022334
building_apartments      0.020764
road_residential_30m     0.020434
has_addr:street          0.015426
building_house           0.014301
building_retail          0.013210
building_kindergarten    0.010486
building_office          0.009411
building_yes             0.008571
parking_size0_90m        0.007285
dtype: float64

In [ ]:
# Hyperparameter sweep -- the paper's original min_samples_split=20/min_impurity_decrease=0.0001
# was kept above deliberately for direct comparability to the Mitte-only baseline, but the
# Path C detour in berlin_mitte.ipynb already showed these are under-regularized even at
# Mitte's ~5k-row scale (best there: min_samples_split=80). At ~265k training rows here,
# skew the grid toward larger min_samples_split too, since more data typically supports (or
# needs) more aggressive pruning before overfitting sets in.
#
# NOTE: this cell needs buildings/X/y/X_train/y_train from the earlier cells in this
# notebook (run those first, or "Run All" from the top) -- it isn't self-contained.
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
param_grid = {
    "min_samples_split": [20, 40, 80, 160, 320],
    "min_impurity_decrease": [0.0001, 0.0005, 0.001, 0.002],
}

base_clf = DecisionTreeClassifier(class_weight={"non_residential": 2.0, "residential": 1.0}, random_state=42)
search = GridSearchCV(base_clf, param_grid, cv=cv, scoring="f1_macro", n_jobs=-1)
search.fit(X_train, y_train)

print("Best CV macro-F1:", search.best_score_)
print("Best params:", search.best_params_)

In [9]:
clf2_berlin_tuned = search.best_estimator_
y_pred_tuned = clf2_berlin_tuned.predict(X_test)

print("=== Tuned ===")
print("Accuracy:", accuracy_score(y_test, y_pred_tuned))
print(classification_report(y_test, y_pred_tuned))
print(confusion_matrix(y_test, y_pred_tuned))

print("\n=== Baseline (paper hyperparameters) ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

=== Tuned ===


Accuracy: 0.9189087175770753


                 precision    recall  f1-score   support

non_residential       0.79      0.80      0.79     13009
    residential       0.95      0.95      0.95     53225

       accuracy                           0.92     66234
      macro avg       0.87      0.87      0.87     66234
   weighted avg       0.92      0.92      0.92     66234



[[10397  2612]
 [ 2759 50466]]

=== Baseline (paper hyperparameters) ===
Accuracy: 0.9189087175770753


                 precision    recall  f1-score   support

non_residential       0.79      0.80      0.79     13009
    residential       0.95      0.95      0.95     53225

       accuracy                           0.92     66234
      macro avg       0.87      0.87      0.87     66234
   weighted avg       0.92      0.92      0.92     66234

